In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import torch
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
import glob
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np

class underwater(Dataset):

  def __init__(self, image_files, masks_files, image_transform = None, mask_transform = None):

    self.image_files = image_files
    self.masks_files = masks_files
    self.image_transform = image_transform
    self.mask_transform = mask_transform

  def __len__(self):

    return len(self.image_files)

  def __getitem__(self, idx):

    image_path = self.image_files[idx]
    mask_path = self.masks_files[idx]

    image = Image.open(image_path).convert('RGB')
    mask = Image.open(mask_path).convert('L')

    if self.image_transform:
        image = self.image_transform(image)

    if self.mask_transform:
        mask = self.mask_transform(mask)

    mask = mask.squeeze(0).long()
    mask = remap_mask(mask)

    return image, mask


In [ ]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

image_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.PILToTensor(),

])



full_path = os.path.join(path, 'dataset')
image_files = glob.glob(f'{full_path}/images/*.jpg')
mask_files = glob.glob(f'{full_path}/masks/*.png')

image_files.sort()
mask_files.sort()

len(image_files), len(mask_files)



In [ ]:
train_images, test_images, train_masks, test_masks = train_test_split(image_files, mask_files, shuffle= True, test_size= 0.2, random_state= 42)

train_dataset = underwater(train_images, train_masks, image_transforms, mask_transforms)
test_dataset = underwater(test_images, test_masks, image_transforms, mask_transforms)

img, mask = train_dataset[0]

print(torch.unique(mask))

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

# Check dataset sizes
print(f"Training Samples: {len(train_dataset)}, Testing Samples: {len(test_dataset)}")

In [ ]:
import matplotlib.pyplot as plt

def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img.numpy().transpose(1, 2, 0)
    img = img * std + mean
    img = np.clip(img, 0, 1)
    return img

for i in range(3):

    img, mask = train_dataset[i]

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].imshow(denormalize(img))
    axes[0].set_title("Image")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="tab20") # I know this method for colorizing the labels before and i used it to get better results.
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")

    plt.show()



In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8,
).to(device)


In [ ]:
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images, masks = images.to(device), masks.to(device)

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
import torch
from torch import nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 5
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")


In [ ]:
import matplotlib.pyplot as plt

plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

In [ ]:
# TO DO

import random
import matplotlib.pyplot as plt
import numpy as np

def denormalize(img):

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img.numpy().transpose(1, 2, 0)
    img = img * std + mean
    img = np.clip(img, 0, 1)
    return img

model.eval()

test_samples = random.sample(range(len(test_dataset)), 5)

for idx in test_samples:
    img, mask = test_dataset[idx]

    with torch.no_grad():
        pred_mask = model(img.unsqueeze(0).to(device))
        pred_mask = torch.argmax(pred_mask, dim=1)

    pred_mask = pred_mask.cpu().squeeze().numpy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(denormalize(img))
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="tab20") # I know this method for colorizing the labels before and I used it to get better results.
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")

    axes[2].imshow(pred_mask, cmap="tab20")
    axes[2].set_title("Predicted Mask")
    axes[2].axis("off")

    plt.show()
